# 15.11 Production Model Serving
A comprehensive guide to deploying ML models at scale using BentoML, KServe, Seldon Core, TensorFlow Serving, TorchServe, Triton, Ray Serve, MLflow, and cloud platforms.

**Topics covered:**
- Serving fundamentals: latency SLAs, throughput, batching, cold starts
- Framework deep-dives: BentoML, KServe, Seldon, TF Serving, TorchServe, Triton, Ray Serve
- Cloud platforms: SageMaker, Vertex AI
- Optimization: caching, async inference, A/B testing
- Monitoring: latency percentiles, drift detection

## 1. Why Model Serving is Hard

Production ML inference faces challenges that batch training never encounters:

**Core challenges:**
- **Latency SLAs**: p99 < 100ms while model inference may take 50-500ms
- **Throughput**: handling 10k+ RPS with limited GPU/CPU resources
- **Cold starts**: container/model initialization adds 5-30s latency spikes
- **GPU utilization**: keeping expensive hardware at 70-90% utilization
- **Batching**: grouping requests to amortize GPU kernel launch overhead

**Little's Law** relates system concurrency, throughput, and latency:

$$L = \lambda W$$

Where:
- $L$ = average number of requests in the system
- $\lambda$ = average arrival rate (requests/sec)
- $W$ = average time a request spends in the system (latency)

**Throughput** formula:

$$\text{Throughput} = \frac{\text{Requests Completed}}{\Delta t}$$

**Latency percentiles**: For a sorted list of $n$ response times $[t_1, t_2, ..., t_n]$, the $p$-th percentile is:

$$t_p = t_{\lceil n \cdot p/100 \rceil}$$

p95 means 95% of requests complete faster than this value a key SLA metric.

In [1]:
import numpy as np
import time

# Simulate request latencies (log-normal distribution is realistic for ML serving)
np.random.seed(42)
n_requests = 10000
latencies_ms = np.random.lognormal(mean=3.9, sigma=0.5, size=n_requests)

p50 = np.percentile(latencies_ms, 50)
p95 = np.percentile(latencies_ms, 95)
p99 = np.percentile(latencies_ms, 99)
p999 = np.percentile(latencies_ms, 99.9)

print(f"Latency Distribution (n={n_requests:,} requests):")
print(f"  p50  (median): {p50:.1f}ms")
print(f"  p95:           {p95:.1f}ms")
print(f"  p99:           {p99:.1f}ms")
print(f"  p99.9:         {p999:.1f}ms")

arrival_rate = 500
avg_latency_sec = p99 / 1000
L = arrival_rate * avg_latency_sec
print(f"\nLittle's Law at {arrival_rate} RPS with p99 latency:")
print(f"  L = lambda * W = {arrival_rate} * {avg_latency_sec:.3f} = {L:.1f} concurrent requests")

batch_sizes = [1, 2, 4, 8, 16, 32, 64]
single_inference_ms = 10
batch_overhead_ms = 5
print("\nBatching throughput analysis:")
for bs in batch_sizes:
    batch_time_ms = single_inference_ms + batch_overhead_ms + bs * 0.5
    throughput = bs / (batch_time_ms / 1000)
    print(f"  Batch={bs:3d}: latency={batch_time_ms:6.1f}ms, throughput={throughput:7.0f} req/s")

Latency Distribution (n=10,000 requests):
  p50  (median): 49.3ms
  p95:           112.3ms
  p99:           158.0ms
  p99.9:         234.3ms

Little's Law at 500 RPS with p99 latency:
  L = lambda * W = 500 * 0.158 = 79.0 concurrent requests

Batching throughput analysis:
  Batch=  1: latency=  15.5ms, throughput=     65 req/s
  Batch=  2: latency=  16.0ms, throughput=    125 req/s
  Batch=  4: latency=  17.0ms, throughput=    235 req/s
  Batch=  8: latency=  19.0ms, throughput=    421 req/s
  Batch= 16: latency=  23.0ms, throughput=    696 req/s
  Batch= 32: latency=  31.0ms, throughput=   1032 req/s
  Batch= 64: latency=  47.0ms, throughput=   1362 req/s


## 2. BentoML

BentoML is a Python-first framework for building and deploying ML services with automatic batching, multi-framework support, and cloud-native packaging.

**Key concepts:**
- `@bentoml.service` decorator that turns a Python class into a deployable service
- **Runners** framework-specific model executors (sklearn, pytorch, tensorflow, etc.)
- **I/O types** `bentoml.io.Image`, `bentoml.io.Text`, `bentoml.io.NumpyNdarray`, `bentoml.io.PandasDataFrame`
- **Adaptive batching** automatically groups concurrent requests into batches
- **Bento** self-contained archive: model + service + dependencies

**Lifecycle:**
1. Save model to BentoML model store: `bentoml.sklearn.save_model()`
2. Define service with `@bentoml.service`
3. Build Bento: `bentoml build`
4. Containerize: `bentoml containerize`
5. Deploy: `bentoml deploy` (BentoCloud) or push Docker image

In [2]:
# BentoML service definition example

bentoml_service_code = '''
import bentoml
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

# 1. Save model to BentoML store
X, y = load_iris(return_X_y=True)
clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)
saved_model = bentoml.sklearn.save_model(
    "iris_classifier", clf,
    signatures={"predict": {"batchable": True, "batch_dim": 0}},
    metadata={"accuracy": 0.97},
)
print(f"Model saved: {saved_model.tag}")

# 2. Define service
iris_runner = bentoml.sklearn.get("iris_classifier:latest").to_runner()
svc = bentoml.Service("iris_service", runners=[iris_runner])

@svc.api(
    input=bentoml.io.NumpyNdarray(shape=(-1, 4), dtype=np.float32),
    output=bentoml.io.NumpyNdarray(),
)
async def classify(input_data):
    return await iris_runner.predict.async_run(input_data)

# 3. Adaptive batching
batched_runner = bentoml.sklearn.get("iris_classifier:latest").to_runner(
    max_batch_size=100,
    max_latency_ms=10,
)
'''
print(bentoml_service_code)

bentofile_yaml = '''
service: "bentoml_service:svc"
labels:
  owner: ml-team
include:
  - "bentoml_service.py"
python:
  packages:
    - scikit-learn==1.3.0
    - numpy>=1.24
docker:
  python_version: "3.10"
'''
print("bentofile.yaml:", bentofile_yaml)

print("""
# Build and deploy commands:
# bentoml build
# bentoml containerize iris_service:latest --platform linux/amd64
# docker run -p 3000:3000 iris_service:latest
# bentoml deploy iris_service:latest --name iris-prod --scaling-min 1 --scaling-max 10
""")


import bentoml
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

# 1. Save model to BentoML store
X, y = load_iris(return_X_y=True)
clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)
saved_model = bentoml.sklearn.save_model(
    "iris_classifier", clf,
    signatures={"predict": {"batchable": True, "batch_dim": 0}},
    metadata={"accuracy": 0.97},
)
print(f"Model saved: {saved_model.tag}")

# 2. Define service
iris_runner = bentoml.sklearn.get("iris_classifier:latest").to_runner()
svc = bentoml.Service("iris_service", runners=[iris_runner])

@svc.api(
    input=bentoml.io.NumpyNdarray(shape=(-1, 4), dtype=np.float32),
    output=bentoml.io.NumpyNdarray(),
)
async def classify(input_data):
    return await iris_runner.predict.async_run(input_data)

# 3. Adaptive batching
batched_runner = bentoml.sklearn.get("iris_classifier:latest").to_runner(
    max_batch_size=100,
    max_latency_ms=10,
)

bentofile

## 3. KServe (formerly KFServing)

KServe provides a Kubernetes-native, serverless inference platform on top of Knative.

**Architecture components:**
- **Predictor** the model server (sklearn, xgboost, tensorflow, pytorch, triton, custom)
- **Transformer** pre/post-processing sidecar
- **Explainer** model explanation server (Alibi, SHAP, LIME)

**Key features:**
- **Canary rollouts** traffic splitting between model versions
- **Multi-model serving** multiple models on single pod (ModelMesh)
- **Scale to zero** Knative-based autoscaling
- **gRPC + REST** Open Inference Protocol (v2)
- **GPU autoscaling** KEDA-based GPU utilization scaling

In [3]:
# KServe InferenceService YAML examples

basic_isvc = '''
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: sklearn-iris
  namespace: kserve-test
  annotations:
    autoscaling.knative.dev/target: "100"
spec:
  predictor:
    sklearn:
      storageUri: "gs://my-bucket/iris-model"
      runtimeVersion: "1.3.0"
      resources:
        requests: {cpu: "100m", memory: "256Mi"}
        limits:   {cpu: "1",    memory: "512Mi"}
'''

canary_isvc = '''
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: sklearn-iris-canary
spec:
  predictor:
    canaryTrafficPercent: 20
    sklearn:
      storageUri: "gs://my-bucket/iris-model-v2"
'''

print("Basic InferenceService:", basic_isvc)
print("Canary (20/80 split):", canary_isvc)

print("""
# Deploy and test:
# kubectl apply -f sklearn-iris.yaml
# kubectl get inferenceservice sklearn-iris -n kserve-test
# INGRESS=$(kubectl get svc istio-ingressgateway -n istio-system -o jsonpath="{.status.loadBalancer.ingress[0].ip}")
# curl -H "Host: sklearn-iris.kserve-test.example.com" http://${INGRESS}/v2/models/sklearn-iris/infer
""")

Basic InferenceService: 
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: sklearn-iris
  namespace: kserve-test
  annotations:
    autoscaling.knative.dev/target: "100"
spec:
  predictor:
    sklearn:
      storageUri: "gs://my-bucket/iris-model"
      runtimeVersion: "1.3.0"
      resources:
        requests: {cpu: "100m", memory: "256Mi"}
        limits:   {cpu: "1",    memory: "512Mi"}

Canary (20/80 split): 
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: sklearn-iris-canary
spec:
  predictor:
    canaryTrafficPercent: 20
    sklearn:
      storageUri: "gs://my-bucket/iris-model-v2"


# Deploy and test:
# kubectl apply -f sklearn-iris.yaml
# kubectl get inferenceservice sklearn-iris -n kserve-test
# INGRESS=$(kubectl get svc istio-ingressgateway -n istio-system -o jsonpath="{.status.loadBalancer.ingress[0].ip}")
# curl -H "Host: sklearn-iris.kserve-test.example.com" http://${INGRESS}/v2/models/sklearn-iris/infer



## 4. Seldon Core

Seldon Core is a production ML serving platform on Kubernetes with powerful inference graph capabilities.

**Graph node types:**
| Type | Purpose |
|------|---------|
| MODEL | Single model predictor |
| ROUTER | Traffic routing (A/B, MAB) |
| COMBINER | Ensemble aggregation |
| TRANSFORMER | Feature transformation |
| OUTPUT_TRANSFORMER | Response post-processing |

In [4]:
# Seldon Core CRD Examples

basic_seldon = '''
apiVersion: machinelearning.seldon.io/v1
kind: SeldonDeployment
metadata:
  name: iris-deployment
  namespace: seldon
spec:
  predictors:
  - name: default
    graph:
      name: iris-model
      type: MODEL
      implementation: SKLEARN_SERVER
      modelUri: gs://my-bucket/iris-sklearn
    replicas: 2
    traffic: 100
'''

ab_test = '''
apiVersion: machinelearning.seldon.io/v1
kind: SeldonDeployment
metadata:
  name: iris-ab-test
spec:
  predictors:
  - name: model-a
    traffic: 70
    graph: {name: model-a, implementation: SKLEARN_SERVER, modelUri: gs://bucket/iris-v1, type: MODEL}
  - name: model-b
    traffic: 30
    graph: {name: model-b, implementation: SKLEARN_SERVER, modelUri: gs://bucket/iris-v2, type: MODEL}
'''

inference_graph = '''
apiVersion: machinelearning.seldon.io/v1
kind: SeldonDeployment
metadata:
  name: iris-pipeline
spec:
  predictors:
  - name: pipeline
    graph:
      name: transformer
      type: TRANSFORMER
      implementation: CUSTOM_CONTAINER
      children:
      - name: iris-model
        type: MODEL
        implementation: SKLEARN_SERVER
        modelUri: gs://bucket/iris-model
        children:
        - name: outlier-detector
          type: OUTPUT_TRANSFORMER
          implementation: CUSTOM_CONTAINER
'''

print("Basic SeldonDeployment:", basic_seldon)
print("A/B Test (70/30):", ab_test)
print("Inference Graph:", inference_graph)

Basic SeldonDeployment: 
apiVersion: machinelearning.seldon.io/v1
kind: SeldonDeployment
metadata:
  name: iris-deployment
  namespace: seldon
spec:
  predictors:
  - name: default
    graph:
      name: iris-model
      type: MODEL
      implementation: SKLEARN_SERVER
      modelUri: gs://my-bucket/iris-sklearn
    replicas: 2
    traffic: 100

A/B Test (70/30): 
apiVersion: machinelearning.seldon.io/v1
kind: SeldonDeployment
metadata:
  name: iris-ab-test
spec:
  predictors:
  - name: model-a
    traffic: 70
    graph: {name: model-a, implementation: SKLEARN_SERVER, modelUri: gs://bucket/iris-v1, type: MODEL}
  - name: model-b
    traffic: 30
    graph: {name: model-b, implementation: SKLEARN_SERVER, modelUri: gs://bucket/iris-v2, type: MODEL}

Inference Graph: 
apiVersion: machinelearning.seldon.io/v1
kind: SeldonDeployment
metadata:
  name: iris-pipeline
spec:
  predictors:
  - name: pipeline
    graph:
      name: transformer
      type: TRANSFORMER
      implementation: CUSTOM_CO

## 5. TensorFlow Serving

TensorFlow Serving is a high-performance serving system for TF models.

**Key features:**
- **SavedModel format**: `model.save(path)` exports graph + weights
- **REST** on port 8501: `POST /v1/models/<name>:predict`
- **gRPC** on port 8500: protobuf-based, lower latency
- **Version policies**: `latest`, `specific`, `all`
- **Batching config**: `max_batch_size`, `batch_timeout_micros`, `num_batch_threads`
- **Warmup records**: TFRecord files in `assets.extra/tf_serving_warmup_requests`

In [5]:
import os, json

batching_config = """max_batch_size { value: 128 }
batch_timeout_micros { value: 10000 }
num_batch_threads { value: 4 }
max_enqueued_batches { value: 1000 }
"""
os.makedirs('/tmp/tf_serving', exist_ok=True)
with open('/tmp/tf_serving/batching_config.txt', 'w') as f:
    f.write(batching_config)
print('batching_config.txt:')
print(batching_config)

model_server_config = {
    'model_config_list': {'config': [
        {'name': 'iris', 'base_path': '/models/iris', 'model_platform': 'tensorflow',
         'model_version_policy': {'latest': {'num_versions': 2}}},
        {'name': 'sentiment', 'base_path': '/models/sentiment', 'model_platform': 'tensorflow',
         'model_version_policy': {'all': {}}}
    ]}
}
print('model_server_config.json:')
print(json.dumps(model_server_config, indent=2))

print("""
# Docker launch:
# docker run -d -p 8500:8500 -p 8501:8501 \\
#   -v /tmp/tf_serving/iris:/models/iris \\
#   -e MODEL_NAME=iris \\
#   tensorflow/serving --enable_batching=true
#
# curl -X POST http://localhost:8501/v1/models/iris:predict \\
#   -H 'Content-Type: application/json' \\
#   -d '{ "instances": [[5.1, 3.5, 1.4, 0.2]] }'
""")

batching_config.txt:
max_batch_size { value: 128 }
batch_timeout_micros { value: 10000 }
num_batch_threads { value: 4 }
max_enqueued_batches { value: 1000 }

model_server_config.json:
{
  "model_config_list": {
    "config": [
      {
        "name": "iris",
        "base_path": "/models/iris",
        "model_platform": "tensorflow",
        "model_version_policy": {
          "latest": {
            "num_versions": 2
          }
        }
      },
      {
        "name": "sentiment",
        "base_path": "/models/sentiment",
        "model_platform": "tensorflow",
        "model_version_policy": {
          "all": {}
        }
      }
    ]
  }
}

# Docker launch:
# docker run -d -p 8500:8500 -p 8501:8501 \
#   -v /tmp/tf_serving/iris:/models/iris \
#   -e MODEL_NAME=iris \
#   tensorflow/serving --enable_batching=true
#
# curl -X POST http://localhost:8501/v1/models/iris:predict \
#   -H 'Content-Type: application/json' \
#   -d '{ "instances": [[5.1, 3.5, 1.4, 0.2]] }'



## 6. TorchServe

TorchServe is PyTorch's native model server.

**Key concepts:**
- `.mar` archive: weights + code + handler bundled together
- **Custom handler**: `initialize`, `preprocess`, `inference`, `postprocess`
- **config.properties**: `batch_size`, `max_batch_delay`, ports
- **Management API** (port 8081): register, scale, unregister models
- **Metrics API** (port 8082): Prometheus-compatible metrics

In [6]:
handler_code = '''
import torch
import numpy as np
from ts.torch_handler.base_handler import BaseHandler

class IrisHandler(BaseHandler):
    def initialize(self, context):
        super().initialize(context)
        self.model.eval()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)

    def preprocess(self, data):
        import json
        rows = []
        for item in data:
            body = item.get("body") or item.get("data")
            if isinstance(body, (bytes, bytearray)):
                body = json.loads(body.decode("utf-8"))
            rows.append(body["features"])
        return torch.tensor(rows, dtype=torch.float32).to(self.device)

    def inference(self, inputs):
        with torch.no_grad():
            return self.model(inputs)

    def postprocess(self, outputs):
        probs = torch.softmax(outputs, dim=-1).cpu().numpy()
        labels = np.argmax(probs, axis=1)
        return [{"label": int(l), "probabilities": p.tolist()} for l, p in zip(labels, probs)]
'''
print(handler_code)

config_props = '''
inference_address=http://0.0.0.0:8080
management_address=http://0.0.0.0:8081
metrics_address=http://0.0.0.0:8082
model_store=/home/model-server/model-store
load_models=all
batch_size=32
max_batch_delay=50
default_workers_per_model=2
'''
print('config.properties:', config_props)

print("""
# Build .mar archive:
# torch-model-archiver --model-name iris --version 1.0 \\
#   --serialized-file /tmp/iris_model.pt \\
#   --handler handler.py \\
#   --export-path /tmp/model_store
#
# torchserve --start --model-store /tmp/model_store --ts-config config.properties
""")


import torch
import numpy as np
from ts.torch_handler.base_handler import BaseHandler

class IrisHandler(BaseHandler):
    def initialize(self, context):
        super().initialize(context)
        self.model.eval()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)

    def preprocess(self, data):
        import json
        rows = []
        for item in data:
            body = item.get("body") or item.get("data")
            if isinstance(body, (bytes, bytearray)):
                body = json.loads(body.decode("utf-8"))
            rows.append(body["features"])
        return torch.tensor(rows, dtype=torch.float32).to(self.device)

    def inference(self, inputs):
        with torch.no_grad():
            return self.model(inputs)

    def postprocess(self, outputs):
        probs = torch.softmax(outputs, dim=-1).cpu().numpy()
        labels = np.argmax(probs, axis=1)
        return [{"label": int(l), "

## 7. Triton Inference Server (NVIDIA)

NVIDIA Triton supports multiple ML frameworks simultaneously with enterprise-grade performance.

**Model repository layout:**
```
model_repository/
  iris_onnx/
    config.pbtxt
    1/
      model.onnx
  iris_trt/
    config.pbtxt
    1/
      model.plan
```

**Backends:**
| Backend | Platform | File |
|---------|----------|------|
| TensorRT | `tensorrt_plan` | model.plan |
| ONNX Runtime | `onnxruntime_onnx` | model.onnx |
| PyTorch | `pytorch_libtorch` | model.pt |
| TensorFlow | `tensorflow_savedmodel` | model.savedmodel/ |
| Python BLS | `python` | model.py |

In [7]:
import os

onnx_config = '''
name: "iris_onnx"
backend: "onnxruntime"
max_batch_size: 128
input  [{ name: "float_input"     data_type: TYPE_FP32 dims: [4] }]
output [{ name: "output_label"    data_type: TYPE_INT64 dims: [1] },
        { name: "output_probability" data_type: TYPE_FP32  dims: [3] }]
dynamic_batching {
  preferred_batch_size: [16, 32, 64]
  max_queue_delay_microseconds: 5000
}
instance_group [{ count: 2  kind: KIND_GPU  gpus: [0] }]
'''

os.makedirs('/tmp/triton/model_repository/iris_onnx/1', exist_ok=True)
with open('/tmp/triton/model_repository/iris_onnx/config.pbtxt', 'w') as f:
    f.write(onnx_config)
print('ONNX config.pbtxt:', onnx_config)

ensemble_config = '''
name: "iris_ensemble"
platform: "ensemble"
max_batch_size: 128
input  [{ name: "RAW_INPUT"   data_type: TYPE_FP32  dims: [4] }]
output [{ name: "FINAL_LABEL" data_type: TYPE_INT64 dims: [1] }]
ensemble_scheduling {
  step [
    { model_name: "preprocessor"  model_version: -1
      input_map  { key: "RAW"          value: "RAW_INPUT" }
      output_map { key: "PREPROCESSED" value: "PREPROCESSED" } },
    { model_name: "iris_onnx" model_version: -1
      input_map  { key: "float_input"  value: "PREPROCESSED" }
      output_map { key: "output_label" value: "FINAL_LABEL" } }
  ]
}
'''
print('Ensemble config.pbtxt:', ensemble_config)

print("""
# Docker launch:
# docker run --gpus all -d -p 8000:8000 -p 8001:8001 -p 8002:8002 \\
#   -v /tmp/triton/model_repository:/models \\
#   nvcr.io/nvidia/tritonserver:24.01-py3 \\
#   tritonserver --model-repository=/models
""")

ONNX config.pbtxt: 
name: "iris_onnx"
backend: "onnxruntime"
max_batch_size: 128
input  [{ name: "float_input"     data_type: TYPE_FP32 dims: [4] }]
output [{ name: "output_label"    data_type: TYPE_INT64 dims: [1] },
        { name: "output_probability" data_type: TYPE_FP32  dims: [3] }]
dynamic_batching {
  preferred_batch_size: [16, 32, 64]
  max_queue_delay_microseconds: 5000
}
instance_group [{ count: 2  kind: KIND_GPU  gpus: [0] }]

Ensemble config.pbtxt: 
name: "iris_ensemble"
platform: "ensemble"
max_batch_size: 128
input  [{ name: "RAW_INPUT"   data_type: TYPE_FP32  dims: [4] }]
output [{ name: "FINAL_LABEL" data_type: TYPE_INT64 dims: [1] }]
ensemble_scheduling {
  step [
    { model_name: "preprocessor"  model_version: -1
      input_map  { key: "RAW"          value: "RAW_INPUT" }
      output_map { key: "PREPROCESSED" value: "PREPROCESSED" } },
    { model_name: "iris_onnx" model_version: -1
      input_map  { key: "float_input"  value: "PREPROCESSED" }
      output_map { k

In [8]:
bls_code = '''
# BLS (Business Logic Scripting), model.py for Python backend
import triton_python_backend_utils as pb_utils

class TritonPythonModel:
    def initialize(self, args):
        self.logger = pb_utils.Logger

    def execute(self, requests):
        responses = []
        for request in requests:
            raw = pb_utils.get_input_tensor_by_name(request, "RAW_INPUT")
            preproc_req = pb_utils.InferenceRequest(
                model_name="preprocessor",
                requested_output_names=["PREPROCESSED"],
                inputs=[raw]
            )
            preproc_resp = preproc_req.exec(decoupled=False)
            preprocessed = pb_utils.get_output_tensor_by_name(preproc_resp, "PREPROCESSED")
            infer_req = pb_utils.InferenceRequest(
                model_name="iris_onnx",
                requested_output_names=["output_label"],
                inputs=[preprocessed]
            )
            infer_resp = infer_req.exec(decoupled=False)
            label = pb_utils.get_output_tensor_by_name(infer_resp, "output_label")
            responses.append(pb_utils.InferenceResponse(output_tensors=[label]))
        return responses

    def finalize(self): pass
'''
print(bls_code)

triton_client_code = '''
import tritonclient.http as httpclient
import numpy as np

client = httpclient.InferenceServerClient(url="localhost:8000")
input_data = np.array([[5.1, 3.5, 1.4, 0.2]], dtype=np.float32)
inputs  = [httpclient.InferInput("float_input", input_data.shape, "FP32")]
outputs = [httpclient.InferRequestedOutput("output_label")]
inputs[0].set_data_from_numpy(input_data)
result = client.infer("iris_onnx", inputs, outputs=outputs)
print(result.as_numpy("output_label"))
'''
print('tritonclient usage:', triton_client_code)


# BLS (Business Logic Scripting), model.py for Python backend
import triton_python_backend_utils as pb_utils

class TritonPythonModel:
    def initialize(self, args):
        self.logger = pb_utils.Logger

    def execute(self, requests):
        responses = []
        for request in requests:
            raw = pb_utils.get_input_tensor_by_name(request, "RAW_INPUT")
            preproc_req = pb_utils.InferenceRequest(
                model_name="preprocessor",
                requested_output_names=["PREPROCESSED"],
                inputs=[raw]
            )
            preproc_resp = preproc_req.exec(decoupled=False)
            preprocessed = pb_utils.get_output_tensor_by_name(preproc_resp, "PREPROCESSED")
            infer_req = pb_utils.InferenceRequest(
                model_name="iris_onnx",
                requested_output_names=["output_label"],
                inputs=[preprocessed]
            )
            infer_resp = infer_req.exec(decoupled=False)
            label = pb_u

## 8. Ray Serve

Ray Serve is a scalable model serving library built on Ray.

**Autoscaling config parameters:**
| Parameter | Description |
|-----------|-------------|
| `min_replicas` | Floor replicas active even at zero traffic |
| `max_replicas` | Ceiling maximum concurrent replicas |
| `target_num_ongoing_requests` | Target in-flight requests per replica |
| `downscale_delay_s` | Seconds before scaling down |
| `upscale_delay_s` | Seconds before scaling up |

**Composable deployments**: Chain Preprocessor -> Model -> Postprocessor using `DeploymentHandle`

In [9]:
ray_serve_code = '''
import ray
from ray import serve
from fastapi import FastAPI
from starlette.requests import Request
import numpy as np

# Basic deployment
@serve.deployment(num_replicas=2, ray_actor_options={"num_cpus": 1})
class IrisClassifier:
    def __init__(self):
        self.classes = ["setosa", "versicolor", "virginica"]

    async def __call__(self, request: Request):
        data = await request.json()
        features = np.array(data["features"])
        pred_idx = int(np.argmax(features[:3]))
        return {"label": self.classes[pred_idx]}

# Autoscaling deployment
@serve.deployment(autoscaling_config={
    "min_replicas": 1, "max_replicas": 10,
    "target_num_ongoing_requests": 5,
    "downscale_delay_s": 300, "upscale_delay_s": 30
})
class AutoscaledClassifier:
    async def run(self, features):
        return {"prediction": 0}

# Composable pipeline using DeploymentHandle
from ray.serve.handle import DeploymentHandle

@serve.deployment()
class Preprocessor:
    MEAN = [5.843, 3.057, 3.758, 1.199]
    STD  = [0.828, 0.435, 1.765, 0.762]
    async def run(self, features):
        arr = (np.array(features) - self.MEAN) / self.STD
        return arr.tolist()

@serve.deployment()
class Pipeline:
    def __init__(self, preprocessor: DeploymentHandle, classifier: DeploymentHandle):
        self.preprocessor = preprocessor
        self.classifier   = classifier

    async def __call__(self, request: Request):
        data = await request.json()
        norm = await self.preprocessor.run.remote(data["features"])
        pred = await self.classifier.run.remote(norm)
        return pred

# FastAPI integration
app = FastAPI()
@serve.deployment()
@serve.ingress(app)
class FastAPIService:
    @app.post("/predict")
    async def predict(self, payload: dict):
        return {"result": "ok"}

ray.init(num_cpus=4, ignore_reinit_error=True)
serve.start(detached=False)
handle = serve.run(Pipeline.bind(Preprocessor.bind(), AutoscaledClassifier.bind()), route_prefix="/iris")
print("Pipeline live at http://localhost:8000/iris")
serve.shutdown()
ray.shutdown()
'''
print(ray_serve_code)


import ray
from ray import serve
from fastapi import FastAPI
from starlette.requests import Request
import numpy as np

# Basic deployment
@serve.deployment(num_replicas=2, ray_actor_options={"num_cpus": 1})
class IrisClassifier:
    def __init__(self):
        self.classes = ["setosa", "versicolor", "virginica"]

    async def __call__(self, request: Request):
        data = await request.json()
        features = np.array(data["features"])
        pred_idx = int(np.argmax(features[:3]))
        return {"label": self.classes[pred_idx]}

# Autoscaling deployment
@serve.deployment(autoscaling_config={
    "min_replicas": 1, "max_replicas": 10,
    "target_num_ongoing_requests": 5,
    "downscale_delay_s": 300, "upscale_delay_s": 30
})
class AutoscaledClassifier:
    async def run(self, features):
        return {"prediction": 0}

# Composable pipeline using DeploymentHandle
from ray.serve.handle import DeploymentHandle

@serve.deployment()
class Preprocessor:
    MEAN = [5.843, 3.057, 

## 9. MLflow Models Serving

**MLflow model flavors:**
| Flavor | Framework |
|--------|-----------|
| `sklearn` | scikit-learn |
| `pytorch` | PyTorch |
| `tensorflow` | TF/Keras |
| `pyfunc` | Any Python |
| `onnx` | ONNX Runtime |

**Serving commands:**
```bash
mlflow models serve -m runs:/<run_id>/model --port 5001 --host 0.0.0.0
mlflow models serve -m models:/MyModel/Production --enable-mlserver
```

**REST API payload formats:**
```json
{"dataframe_split": {"columns": ["f1"], "data": [[1.0]]}}
{"inputs": [[1.0, 2.0, 3.0]]}
```

In [10]:
import mlflow
import mlflow.sklearn
import mlflow.pyfunc
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from mlflow.models.signature import infer_signature

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)
signature = infer_signature(X, clf.predict(X))

with mlflow.start_run(run_name='sklearn-iris') as run:
    mlflow.sklearn.log_model(
        sk_model=clf, artifact_path='model',
        signature=signature, input_example=X.iloc[:3],
        registered_model_name='IrisClassifier',
    )
    run_id = run.info.run_id
    print(f'Logged model, run_id={run_id}')

class ThresholdedClassifier(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import joblib
        self.model = joblib.load(context.artifacts['sklearn_model'])

    def predict(self, context, model_input):
        proba = self.model.predict_proba(model_input)
        return pd.DataFrame({'prediction': np.argmax(proba, axis=1), 'max_prob': proba.max(axis=1)})

import joblib, os, tempfile
tmpdir = tempfile.mkdtemp()
model_path = os.path.join(tmpdir, 'rf_model.pkl')
joblib.dump(clf, model_path)

with mlflow.start_run(run_name='pyfunc-custom') as run:
    mlflow.pyfunc.log_model(
        artifact_path='model',
        python_model=ThresholdedClassifier(),
        artifacts={'sklearn_model': model_path},
        signature=infer_signature(X, pd.DataFrame({'prediction': [0], 'max_prob': [0.9]})),
        registered_model_name='ThresholdedIrisClassifier',
    )
    print(f'Custom pyfunc model logged, run_id={run.info.run_id}')

print(f'\nServe: mlflow models serve -m runs:/{run_id}/model --port 5001')
print('MLServer: mlflow models serve -m models:/IrisClassifier/1 --enable-mlserver --port 8080')

2026/06/19 16:29:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/19 16:30:31 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Successfully registered model 'IrisClassifier'.
Created version '1' of model 'IrisClassifier'.


Logged model, run_id=5f07726f3fe84d76a81c0ba99b53a230


/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/19 16:30:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/19 16:30:33 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3341: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


Successfully registered model 'ThresholdedIrisClassifier'.
Created version '1' of model 'ThresholdedIrisClassifier'.


Custom pyfunc model logged, run_id=dbe06dd01e3d4360a6810739c2ce5276

Serve: mlflow models serve -m runs:/5f07726f3fe84d76a81c0ba99b53a230/model --port 5001
MLServer: mlflow models serve -m models:/IrisClassifier/1 --enable-mlserver --port 8080


## 10. Inference Optimization Patterns

### Batching Types
| Type | Mechanism | Best For |
|------|-----------|----------|
| **Static** | Fixed batch size | Offline / throughput-optimized |
| **Dynamic** | Collect until timeout OR max size | General serving |
| **Continuous** | Replace finished sequences mid-flight | LLM token streaming |

### Response Caching
Cache key = `hashlib.sha256(json.dumps(input))` TTL-based Redis expiry

### Multi-threading vs Multi-process
| | ThreadPoolExecutor | ProcessPoolExecutor |
|-|--------------------|-----------------------|
| GIL | Shared | Separate process |
| Best for | I/O-bound | CPU-bound numpy/preprocessing |

In [11]:
import asyncio, hashlib, json, time
import numpy as np
from typing import Any, Dict, Optional

class PredictionCache:
    def __init__(self, ttl_seconds: int = 3600):
        self.ttl = ttl_seconds
        self._local: Dict[str, Any] = {}

    def _make_key(self, inputs: Any) -> str:
        serialized = json.dumps(inputs, sort_keys=True, default=str).encode()
        return 'pred:' + hashlib.sha256(serialized).hexdigest()

    def get(self, inputs: Any) -> Optional[Any]:
        return self._local.get(self._make_key(inputs))

    def set(self, inputs: Any, prediction: Any) -> None:
        self._local[self._make_key(inputs)] = prediction

cache = PredictionCache()
sample = [[1.2, 3.4, 5.6, 7.8]]
cache.set(sample, {'label': 2, 'proba': [0.05, 0.10, 0.85]})
print('Cache hit:', cache.get(sample))
print('Cache miss:', cache.get([[9.9, 9.9, 9.9, 9.9]]))

def dummy_model(x: np.ndarray) -> np.ndarray:
    time.sleep(0.005)
    return np.sum(x, axis=1)

N = 16
t0 = time.perf_counter()
for _ in range(N):
    dummy_model(np.random.rand(1, 4))
single_ms = (time.perf_counter() - t0) / N * 1000

t0 = time.perf_counter()
for _ in range(N // 8):
    dummy_model(np.random.rand(8, 4))
batch_ms = (time.perf_counter() - t0) / N * 1000

print(f'Avg latency/sample, single: {single_ms:.2f}ms | batched(8): {batch_ms:.2f}ms')
print(f'Throughput gain: {single_ms/batch_ms:.1f}x')

Cache hit: {'label': 2, 'proba': [0.05, 0.1, 0.85]}
Cache miss: None


Avg latency/sample, single: 5.87ms | batched(8): 0.66ms
Throughput gain: 8.9x


In [12]:
import time, math
import numpy as np
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def warmup_model(model_fn, input_shape=(1, 4), n_warmup=10):
    print(f'Warming up with {n_warmup} dummy requests...')
    dummy = np.zeros(input_shape, dtype=np.float32)
    t0 = time.perf_counter()
    for _ in range(n_warmup):
        _ = model_fn(dummy)
    elapsed = (time.perf_counter() - t0) * 1000
    print(f'Warmup done in {elapsed:.1f}ms ({elapsed/n_warmup:.1f}ms/req)')

def toy_model(x):
    time.sleep(0.002)
    return float(np.sum(x))

warmup_model(toy_model, input_shape=(1, 4), n_warmup=5)

def cpu_task(n):
    return sum(math.sqrt(i) for i in range(n))

N_TASKS, TASK_SIZE = 8, 50_000
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(cpu_task, [TASK_SIZE]*N_TASKS))
thread_ms = (time.perf_counter() - t0)*1000

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as ex:
    list(ex.map(cpu_task, [TASK_SIZE]*N_TASKS))
proc_ms = (time.perf_counter() - t0)*1000

print(f'ThreadPool  (CPU-bound, {N_TASKS} tasks): {thread_ms:.1f}ms')
print(f'ProcessPool (CPU-bound, {N_TASKS} tasks): {proc_ms:.1f}ms')

Warming up with 5 dummy requests...
Warmup done in 16.7ms (3.3ms/req)


ThreadPool  (CPU-bound, 8 tasks): 127.8ms
ProcessPool (CPU-bound, 8 tasks): 529.1ms


## 11. A/B Testing for Models

**Traffic splitting**: percentage-based routing per request

**Shadow mode**: run challenger on every request, log its prediction, but only serve the champion response.

**Champion-challenger pattern:**
1. Run A/B for N requests
2. Compute metric distributions
3. Mann-Whitney U test for significance
4. Promote if `p < 0.05` AND improvement > threshold

**Gradual rollout:** `1% -> 5% -> 20% -> 50% -> 100%`

**PSI for drift:**
$$\text{PSI} = \sum_{i=1}^{N} (A_i - E_i) \cdot \ln\left(\frac{A_i}{E_i}\right)$$

In [13]:
import random, time
import numpy as np
from typing import Dict, List
from scipy import stats
from dataclasses import dataclass, field

class ABTestRouter:
    def __init__(self, model_a, model_b, traffic_split=None, shadow_mode=False):
        self.model_a = model_a
        self.model_b = model_b
        self.traffic_split = traffic_split or {'a': 0.80, 'b': 0.20}
        self.shadow_mode = shadow_mode
        self.logs: List[Dict] = []

    def predict(self, inputs):
        r = random.random()
        if self.shadow_mode:
            t0 = time.perf_counter()
            pred = self.model_a(inputs)
            lat = (time.perf_counter() - t0)*1000
            shadow = self.model_b(inputs)
            self.logs.append({'model': 'a', 'pred': pred, 'latency_ms': lat, 'shadow': shadow})
            return pred
        route_b = r >= self.traffic_split['a']
        model = self.model_b if route_b else self.model_a
        variant = 'b' if route_b else 'a'
        t0 = time.perf_counter()
        pred = model(inputs)
        lat = (time.perf_counter() - t0)*1000
        self.logs.append({'model': variant, 'pred': pred, 'latency_ms': lat})
        return pred

@dataclass
class RolloutScheduler:
    schedule: List[float] = field(default_factory=lambda: [0.01, 0.05, 0.20, 0.50, 1.00])
    stage: int = 0

    @property
    def challenger_pct(self): return self.schedule[self.stage]

    def advance(self):
        if self.stage < len(self.schedule)-1:
            self.stage += 1
            print(f'Rollout -> {self.challenger_pct*100:.0f}% challenger')

model_a = lambda x: float(np.dot(x, [0.1, 0.2, 0.3, 0.4]))
model_b = lambda x: float(np.dot(x, [0.11, 0.21, 0.31, 0.41])) + 0.05

router = ABTestRouter(model_a, model_b, traffic_split={'a': 0.8, 'b': 0.2})
inp = np.random.rand(4).tolist()
for _ in range(100):
    router.predict(inp)

counts = {'a': sum(1 for l in router.logs if l['model']=='a'),
          'b': sum(1 for l in router.logs if l['model']=='b')}
print(f'Traffic split: A={counts["a"]} B={counts["b"]} (target 80/20)')

rs = RolloutScheduler()
for _ in range(5): rs.advance()

Traffic split: A=83 B=17 (target 80/20)
Rollout -> 5% challenger
Rollout -> 20% challenger
Rollout -> 50% challenger
Rollout -> 100% challenger


In [14]:
import numpy as np
from scipy import stats

def champion_challenger_decision(metrics_champion, metrics_challenger,
                                  alpha=0.05, min_improvement_pct=2.0, higher_is_better=True):
    stat, p = stats.mannwhitneyu(metrics_champion, metrics_challenger, alternative='two-sided')
    mean_c  = np.mean(metrics_champion)
    mean_ch = np.mean(metrics_challenger)
    improvement = ((mean_ch - mean_c) / (abs(mean_c) + 1e-9)) * 100
    if not higher_is_better:
        improvement = -improvement
    promote = (p < alpha) and (improvement >= min_improvement_pct)
    return {'mean_champion': round(mean_c,4), 'mean_challenger': round(mean_ch,4),
            'improvement_pct': round(improvement,2), 'p_value': round(p,5),
            'significant': p < alpha, 'verdict': 'PROMOTE' if promote else 'KEEP CHAMPION'}

np.random.seed(42)
acc_a = np.random.normal(0.82, 0.04, 500).tolist()
acc_b = np.random.normal(0.855, 0.04, 500).tolist()

result = champion_challenger_decision(acc_a, acc_b, higher_is_better=True)
print('Champion-Challenger Decision:')
for k, v in result.items():
    print(f'  {k}: {v}')

Champion-Challenger Decision:
  mean_champion: 0.8203
  mean_challenger: 0.8563
  improvement_pct: 4.39
  p_value: 0.0
  significant: True
  verdict: PROMOTE


## 12. SageMaker Endpoints

| Type | Latency | Max Payload | Best For |
|------|---------|-------------|----------|
| Real-time | <100ms | 6 MB | Low-latency APIs |
| Async | Seconds | 1 GB | Large inputs |
| Serverless | ~1-3s | 4 MB | Spiky traffic |
| Multi-model (MME) | <100ms | 6 MB | Many models |

**Auto-scaling**: `TargetTrackingScaling` on `SageMakerVariantInvocationsPerInstance`

In [15]:
sagemaker_code = '''
import boto3, sagemaker
from sagemaker.model import Model
from sagemaker.async_inference import AsyncInferenceConfig
from sagemaker.serverless import ServerlessInferenceConfig

ROLE    = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
BUCKET  = "my-sagemaker-bucket"
IMAGE   = "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"
MODEL_URI = f"s3://{BUCKET}/models/iris/model.tar.gz"

model = Model(image_uri=IMAGE, model_data=MODEL_URI, role=ROLE)

# Real-time endpoint
predictor = model.deploy(initial_instance_count=2, instance_type="ml.m5.large",
                          endpoint_name="iris-realtime")

# Async endpoint
async_cfg = AsyncInferenceConfig(
    output_path=f"s3://{BUCKET}/async-output/",
    notification_config={"SuccessTopic": "arn:aws:sns:...:inference-success"}
)
async_predictor = model.deploy(
    initial_instance_count=1, instance_type="ml.m5.xlarge",
    async_inference_config=async_cfg
)

# Serverless endpoint
sl_cfg = ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=20)
sl_predictor = model.deploy(serverless_inference_config=sl_cfg)

# Auto-scaling
aas = boto3.client("application-autoscaling")
resource_id = "endpoint/iris-realtime/variant/AllTraffic"
aas.register_scalable_target(
    ServiceNamespace="sagemaker", ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=1, MaxCapacity=8
)
aas.put_scaling_policy(
    PolicyName="iris-target-tracking", ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 70.0,
        "PredefinedMetricSpecification": {"PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"},
        "ScaleOutCooldown": 60, "ScaleInCooldown": 300
    }
)
'''
print(sagemaker_code)


import boto3, sagemaker
from sagemaker.model import Model
from sagemaker.async_inference import AsyncInferenceConfig
from sagemaker.serverless import ServerlessInferenceConfig

ROLE    = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
BUCKET  = "my-sagemaker-bucket"
IMAGE   = "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"
MODEL_URI = f"s3://{BUCKET}/models/iris/model.tar.gz"

model = Model(image_uri=IMAGE, model_data=MODEL_URI, role=ROLE)

# Real-time endpoint
predictor = model.deploy(initial_instance_count=2, instance_type="ml.m5.large",
                          endpoint_name="iris-realtime")

# Async endpoint
async_cfg = AsyncInferenceConfig(
    output_path=f"s3://{BUCKET}/async-output/",
    notification_config={"SuccessTopic": "arn:aws:sns:...:inference-success"}
)
async_predictor = model.deploy(
    initial_instance_count=1, instance_type="ml.m5.xlarge",
    async_inference_config=async_cfg
)

# Serverless endpoint
sl_cfg = Serverles

In [16]:
mme_code = '''
from sagemaker.multidatamodel import MultiDataModel
import boto3, json

MME_S3_PREFIX = "s3://my-bucket/mme-models/"
mme = MultiDataModel(
    name="multi-model-endpoint",
    model_data_prefix=MME_S3_PREFIX,
    image_uri=IMAGE, role=ROLE
)
predictor = mme.deploy(initial_instance_count=2, instance_type="ml.m5.xlarge")
mme.add_model(model_data_source="s3://bucket/iris_v1.tar.gz", model_data_path="iris_v1.tar.gz")
mme.add_model(model_data_source="s3://bucket/iris_v2.tar.gz", model_data_path="iris_v2.tar.gz")

pred_v1 = predictor.predict(data=[[5.1, 3.5, 1.4, 0.2]], target_model="iris_v1.tar.gz")

runtime = boto3.client("sagemaker-runtime")
resp = runtime.invoke_endpoint(
    EndpointName="iris-realtime",
    ContentType="application/json",
    Body=json.dumps([[5.1, 3.5, 1.4, 0.2]]).encode()
)
prediction = json.loads(resp["Body"].read())

aresp = runtime.invoke_endpoint_async(
    EndpointName="iris-async", ContentType="application/json",
    InputLocation="s3://bucket/input/sample.json"
)
print("Async output path:", aresp["OutputLocation"])
'''
print(mme_code)


from sagemaker.multidatamodel import MultiDataModel
import boto3, json

MME_S3_PREFIX = "s3://my-bucket/mme-models/"
mme = MultiDataModel(
    name="multi-model-endpoint",
    model_data_prefix=MME_S3_PREFIX,
    image_uri=IMAGE, role=ROLE
)
predictor = mme.deploy(initial_instance_count=2, instance_type="ml.m5.xlarge")
mme.add_model(model_data_source="s3://bucket/iris_v1.tar.gz", model_data_path="iris_v1.tar.gz")
mme.add_model(model_data_source="s3://bucket/iris_v2.tar.gz", model_data_path="iris_v2.tar.gz")

pred_v1 = predictor.predict(data=[[5.1, 3.5, 1.4, 0.2]], target_model="iris_v1.tar.gz")

runtime = boto3.client("sagemaker-runtime")
resp = runtime.invoke_endpoint(
    EndpointName="iris-realtime",
    ContentType="application/json",
    Body=json.dumps([[5.1, 3.5, 1.4, 0.2]]).encode()
)
prediction = json.loads(resp["Body"].read())

aresp = runtime.invoke_endpoint_async(
    EndpointName="iris-async", ContentType="application/json",
    InputLocation="s3://bucket/input/sample.jso

## 13. Vertex AI Prediction

**Machine types:**
- Dedicated: `n1-standard-4`, `n1-highmem-8`, `a2-highgpu-1g` (A100)
- Shared: serverless, pay-per-compute-unit

**Prediction modes:**
- **Online**: real-time REST/gRPC, auto-scaling replicas, traffic splitting
- **Batch**: GCS/BigQuery input, managed job, cost-efficient for large datasets

**Model Monitoring**: skew/drift detection per feature

**Explainable AI**: SHAP-based feature attributions (sampled Shapley, integrated gradients, XRAI)

In [17]:
vertex_code = '''
from google.cloud import aiplatform

PROJECT_ID = "my-gcp-project"
LOCATION   = "us-central1"
BUCKET     = f"gs://{PROJECT_ID}-vertex"
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET)

model_v1 = aiplatform.Model.upload(
    display_name="fraud-detector-v1",
    artifact_uri=f"{BUCKET}/models/fraud_v1",
    serving_container_image_uri="us-docker.pkg.dev/my-gcp-project/ml-repo/fraud-serving:v1",
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
    serving_container_ports=[8080],
)
endpoint = aiplatform.Endpoint.create(display_name="fraud-detector-endpoint")

# Deploy v1 (80% traffic)
endpoint.deploy(model=model_v1, deployed_model_display_name="fraud-v1",
                machine_type="n1-standard-4", min_replica_count=1, max_replica_count=5,
                traffic_percentage=80, sync=True)

# Canary v2 (20%)
model_v2 = aiplatform.Model.upload(
    display_name="fraud-detector-v2",
    artifact_uri=f"{BUCKET}/models/fraud_v2",
    serving_container_image_uri="us-docker.pkg.dev/my-gcp-project/ml-repo/fraud-serving:v2",
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
    serving_container_ports=[8080],
)
endpoint.deploy(model=model_v2, deployed_model_display_name="fraud-v2",
                machine_type="n1-standard-4", min_replica_count=1, max_replica_count=3,
                traffic_percentage=20, sync=True)

instances = [
    {"amount": 250.0, "merchant_category": 5411, "hour": 14, "distance_km": 2.3},
    {"amount": 9999.0, "merchant_category": 7995, "hour": 3, "distance_km": 8500.0},
]
response = endpoint.predict(instances=instances)
for pred in response.predictions:
    print(pred)
'''
print(vertex_code)


from google.cloud import aiplatform

PROJECT_ID = "my-gcp-project"
LOCATION   = "us-central1"
BUCKET     = f"gs://{PROJECT_ID}-vertex"
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET)

model_v1 = aiplatform.Model.upload(
    display_name="fraud-detector-v1",
    artifact_uri=f"{BUCKET}/models/fraud_v1",
    serving_container_image_uri="us-docker.pkg.dev/my-gcp-project/ml-repo/fraud-serving:v1",
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
    serving_container_ports=[8080],
)
endpoint = aiplatform.Endpoint.create(display_name="fraud-detector-endpoint")

# Deploy v1 (80% traffic)
endpoint.deploy(model=model_v1, deployed_model_display_name="fraud-v1",
                machine_type="n1-standard-4", min_replica_count=1, max_replica_count=5,
                traffic_percentage=80, sync=True)

# Canary v2 (20%)
model_v2 = aiplatform.Model.upload(
    display_name="fraud-detector-v2",
    artifact_uri=f"{BUCKET}/mode

In [18]:
batch_code = '''
from google.cloud import aiplatform

PROJECT_ID = "my-gcp-project"
BUCKET     = f"gs://{PROJECT_ID}-vertex"
aiplatform.init(project=PROJECT_ID, location="us-central1")

model = aiplatform.Model("projects/my-gcp-project/locations/us-central1/models/123")
batch_job = model.batch_predict(
    job_display_name="fraud-batch-scoring",
    instances_format="jsonl",
    predictions_format="jsonl",
    gcs_source=[f"{BUCKET}/batch_inputs/transactions.jsonl"],
    gcs_destination_prefix=f"{BUCKET}/batch_outputs/",
    machine_type="n1-standard-8",
    starting_replica_count=2, max_replica_count=10,
    sync=False,
)
print(f"Batch job: {batch_job.resource_name}")

endpoints = aiplatform.Endpoint.list(
    filter="display_name=\"fraud*\"", order_by="create_time desc"
)
for ep in endpoints:
    print(f"Endpoint: {ep.display_name}")
'''
print(batch_code)


from google.cloud import aiplatform

PROJECT_ID = "my-gcp-project"
BUCKET     = f"gs://{PROJECT_ID}-vertex"
aiplatform.init(project=PROJECT_ID, location="us-central1")

model = aiplatform.Model("projects/my-gcp-project/locations/us-central1/models/123")
batch_job = model.batch_predict(
    job_display_name="fraud-batch-scoring",
    instances_format="jsonl",
    predictions_format="jsonl",
    gcs_source=[f"{BUCKET}/batch_inputs/transactions.jsonl"],
    gcs_destination_prefix=f"{BUCKET}/batch_outputs/",
    machine_type="n1-standard-8",
    starting_replica_count=2, max_replica_count=10,
    sync=False,
)
print(f"Batch job: {batch_job.resource_name}")

endpoints = aiplatform.Endpoint.list(
    filter="display_name="fraud*"", order_by="create_time desc"
)
for ep in endpoints:
    print(f"Endpoint: {ep.display_name}")



## 14. gRPC vs REST

### Performance Comparison

| Metric | REST/JSON | gRPC/protobuf |
|--------|-----------|---------------|
| Payload size | 1x (baseline) | 0.1-0.3x |
| Serialization | ~50 us | ~5 us |
| p50 latency | ~2 ms | ~0.5 ms |
| p99 latency | ~20 ms | ~5 ms |

### Protocol Buffers
- Binary encoding: ~3-10x smaller than JSON
- Schema-enforced at compile time
- HTTP/2: multiplexing, header compression (HPACK)

### When to use each
- **gRPC**: internal microservices, >10K RPS, streaming
- **REST**: public APIs, browser clients, simple integrations

In [19]:
import json, timeit, struct

PROTO_DEF = '''
syntax = "proto3";
package prediction;

message PredictRequest {
    repeated float features   = 1;
    string  model_name        = 2;
    int32   top_k             = 3;
}
message PredictResponse {
    repeated float probabilities   = 1;
    int32          predicted_class = 2;
    float          latency_ms      = 3;
}
service PredictionService {
    rpc Predict      (PredictRequest) returns (PredictResponse);
    rpc BatchPredict (PredictRequest) returns (stream PredictResponse);
}
'''
print('Proto definition:', PROTO_DEF)

payload  = {"features": [0.5, 1.2, -0.3, 0.8, 2.1, 0.0, -1.5, 3.3, 0.7, 1.9],
            "model_name": "fraud-detector-v2", "top_k": 3}
features = payload['features']
N = 100_000

json_ser   = timeit.timeit(lambda: json.dumps(payload).encode(), number=N)
json_bytes = json.dumps(payload).encode()
json_deser = timeit.timeit(lambda: json.loads(json_bytes), number=N)

proto_ser   = timeit.timeit(lambda: struct.pack(f'{len(features)}f', *features), number=N)
proto_bytes = struct.pack(f'{len(features)}f', *features)
proto_deser = timeit.timeit(lambda: struct.unpack(f'{len(features)}f', proto_bytes), number=N)

print(f'Payload: JSON={len(json_bytes)}B  proto-proxy={len(proto_bytes)}B ({len(proto_bytes)/len(json_bytes):.2f}x)')
print(f'Serialize:   JSON={json_ser*1e6/N:.2f}us  proto={proto_ser*1e6/N:.2f}us (speedup {json_ser/proto_ser:.1f}x)')
print(f'Deserialize: JSON={json_deser*1e6/N:.2f}us proto={proto_deser*1e6/N:.2f}us (speedup {json_deser/proto_deser:.1f}x)')

Proto definition: 
syntax = "proto3";
package prediction;

message PredictRequest {
    repeated float features   = 1;
    string  model_name        = 2;
    int32   top_k             = 3;
}
message PredictResponse {
    repeated float probabilities   = 1;
    int32          predicted_class = 2;
    float          latency_ms      = 3;
}
service PredictionService {
    rpc Predict      (PredictRequest) returns (PredictResponse);
    rpc BatchPredict (PredictRequest) returns (stream PredictResponse);
}



Payload: JSON=113B  proto-proxy=40B (0.35x)
Serialize:   JSON=23.22us  proto=3.89us (speedup 6.0x)
Deserialize: JSON=23.80us proto=3.32us (speedup 7.2x)


In [20]:
import random, statistics

def simulate_latency(protocol, n=500):
    random.seed(42)
    lats = []
    for _ in range(n):
        if protocol == 'grpc':
            base  = random.gauss(0.5, 0.15)
            spike = random.expovariate(1/0.3)
            lat   = max(0.1, base) + (spike if random.random() < 0.02 else 0)
        else:
            base  = random.gauss(2.0, 0.5)
            spike = random.expovariate(1/5.0)
            lat   = max(0.3, base) + (spike if random.random() < 0.05 else 0)
        lats.append(lat)
    return sorted(lats)

print('Protocol Latency Comparison (simulated, 500 requests):')
print(f"{'Protocol':8s} {'mean':>8s} {'p50':>8s} {'p95':>8s} {'p99':>8s}")
for proto in ['grpc', 'rest']:
    lats = simulate_latency(proto)
    print(f"{proto.upper():8s} {statistics.mean(lats):>7.2f}ms "
          f"{lats[int(.50*len(lats))]:>7.2f}ms "
          f"{lats[int(.95*len(lats))]:>7.2f}ms "
          f"{lats[int(.99*len(lats))]:>7.2f}ms")

Protocol Latency Comparison (simulated, 500 requests):
Protocol     mean      p50      p95      p99
GRPC        0.50ms    0.50ms    0.77ms    0.85ms
REST        2.19ms    2.02ms    3.12ms    7.43ms


## 15. Serving Monitoring

**Key metrics:**
- **Latency**: p50/p95/p99 via Prometheus histogram
  - PromQL: `histogram_quantile(0.99, rate(request_latency_seconds_bucket[5m]))`
- **Throughput**: QPS via `rate(requests_total[1m])`
- **Error rate**: `rate(requests_total{status=~"5.."}[5m]) / rate(requests_total[5m])`
- **GPU utilization**: DCGM exporter -> `DCGM_FI_DEV_GPU_UTIL`

**PSI (Population Stability Index):**
$$\text{PSI} = \sum_{i=1}^{N} (A_i - E_i) \cdot \ln\left(\frac{A_i}{E_i}\right)$$

PSI < 0.1: OK | 0.1-0.2: WARNING | > 0.2: ALERT trigger retraining

In [21]:
from prometheus_client import Counter, Histogram, Gauge, CollectorRegistry
import random

REGISTRY = CollectorRegistry()

REQUEST_LATENCY = Histogram(
    'request_latency_seconds', 'Request latency',
    labelnames=['endpoint', 'method', 'status_code'],
    buckets=[0.001, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0, 2.5, 5.0],
    registry=REGISTRY,
)
REQUESTS_TOTAL = Counter(
    'requests_total', 'Total requests',
    labelnames=['endpoint', 'method', 'status_code'],
    registry=REGISTRY,
)
GPU_UTIL = Gauge('gpu_utilization_percent', 'GPU utilization',
                  labelnames=['gpu_index'], registry=REGISTRY)

random.seed(0)
for _ in range(50):
    lat = random.lognormvariate(-3.0, 0.8)
    sc  = '200' if random.random() > 0.02 else '500'
    REQUEST_LATENCY.labels('/predict', 'POST', sc).observe(lat)
    REQUESTS_TOTAL.labels('/predict', 'POST', sc).inc()
GPU_UTIL.labels('0').set(72.5)

print('Metrics registered and populated.')
print('\nPromQL reference:')
promql = [
    ('p99 latency',    'histogram_quantile(0.99, rate(request_latency_seconds_bucket{endpoint="/predict"}[5m]))'),
    ('QPS',            'rate(requests_total{endpoint="/predict",status_code="200"}[1m])'),
    ('Error rate',     'rate(requests_total{status_code=~"5.."}[5m]) / rate(requests_total[5m])'),
    ('GPU util',       'gpu_utilization_percent{gpu_index="0"}'),
    ('Alert p99>200ms','histogram_quantile(0.99, rate(request_latency_seconds_bucket[5m])) > 0.200'),
]
for label, q in promql:
    print(f'  [{label}]\n  {q}')

Metrics registered and populated.

PromQL reference:
  [p99 latency]
  histogram_quantile(0.99, rate(request_latency_seconds_bucket{endpoint="/predict"}[5m]))
  [QPS]
  rate(requests_total{endpoint="/predict",status_code="200"}[1m])
  [Error rate]
  rate(requests_total{status_code=~"5.."}[5m]) / rate(requests_total[5m])
  [GPU util]
  gpu_utilization_percent{gpu_index="0"}
  [Alert p99>200ms]
  histogram_quantile(0.99, rate(request_latency_seconds_bucket[5m])) > 0.200


In [22]:
import numpy as np
import json
from scipy.stats import entropy

def compute_psi(expected, actual, n_bins=10, eps=1e-6):
    bins = np.percentile(expected, np.linspace(0, 100, n_bins+1))
    bins[0] -= eps; bins[-1] += eps
    E = np.histogram(expected, bins=bins)[0].astype(float) + eps
    A = np.histogram(actual,   bins=bins)[0].astype(float) + eps
    E /= E.sum(); A /= A.sum()
    return float(np.sum((A - E) * np.log(A / E)))

def compute_kl(p, q, n_bins=50, eps=1e-6):
    lo = min(p.min(), q.min()) - eps
    hi = max(p.max(), q.max()) + eps
    bins = np.linspace(lo, hi, n_bins+1)
    ph = np.histogram(p, bins)[0].astype(float) + eps
    qh = np.histogram(q, bins)[0].astype(float) + eps
    ph /= ph.sum(); qh /= qh.sum()
    return float(entropy(ph, qh))

np.random.seed(42)
ref     = np.random.normal(0.0, 1.0, 10_000)
stable  = np.random.normal(0.05, 1.05, 2_000)
drifted = np.random.normal(1.2, 1.5, 2_000)

for label, dist in [('stable', stable), ('drifted', drifted)]:
    psi = compute_psi(ref, dist)
    kl  = compute_kl(ref, dist)
    status = 'OK' if psi < 0.1 else ('WARN' if psi < 0.2 else 'ALERT, retrain')
    print(f'{label:8s}: PSI={psi:.4f}  KL={kl:.4f}  [{status}]')

stable  : PSI=0.0054  KL=0.0317  [OK]
drifted : PSI=0.9209  KL=0.4947  [ALERT, retrain]


## 16. Framework Selection Guide

| Framework | Best For | Scale | GPU | Cloud-Native |
|-----------|----------|-------|-----|--------------|
| BentoML | Python-first, quick deploy | Medium | Yes | BentoCloud |
| KServe | Kubernetes, serverless | High | Yes | Any K8s |
| Seldon Core | Inference graphs, A/B | High | Yes | Any K8s |
| TF Serving | TensorFlow models | High | Yes | Any |
| TorchServe | PyTorch models | High | Yes | Any |
| Triton | Multi-framework, GPU opt | Very High | Yes | Any |
| Ray Serve | Python ML, composable | High | Yes | Anyscale |
| MLflow | Experiment tracking + serve | Low-Med | Limited | MLflow Cloud |
| SageMaker | AWS managed, no K8s ops | Very High | Yes | AWS |
| Vertex AI | GCP managed, AutoML | Very High | Yes | GCP |

### Decision Flowchart
```
On AWS?          -> SageMaker
On GCP?          -> Vertex AI
No K8s?          -> BentoML or Ray Serve
Max GPU perf?    -> Triton
A/B graphs?      -> Seldon Core
TensorFlow?      -> TF Serving
PyTorch?         -> TorchServe
Serverless K8s?  -> KServe
Tracking needed? -> MLflow
```

### Key Takeaways
- Start simple: BentoML or MLflow for prototyping
- GPU at scale: Triton is the gold standard
- Managed vs self-managed: SageMaker/Vertex AI eliminate K8s ops at cost of lock-in
- Monitor everything: latency, QPS, error rate, GPU util, feature drift
- Traffic splitting is the safest way to validate model updates in production

In [23]:
from dataclasses import dataclass, field
from typing import Optional, List

@dataclass
class ServingFramework:
    name: str; best_for: str; scale: str; gpu: bool
    cloud_native: str; requires_k8s: bool; cloud: Optional[str]
    frameworks: list; min_ops: bool

FRAMEWORKS = [
    ServingFramework('BentoML',     'Python-first, quick deploy',        'Medium',    True,  'BentoCloud',   False, None,  ['any'],                     False),
    ServingFramework('KServe',      'Kubernetes, serverless',            'High',      True,  'Any K8s',      True,  None,  ['tensorflow','pytorch'],    False),
    ServingFramework('Seldon Core', 'Inference graphs, A/B testing',     'High',      True,  'Any K8s',      True,  None,  ['any'],                     False),
    ServingFramework('TF Serving',  'TensorFlow models',                 'High',      True,  'Any',          False, None,  ['tensorflow'],              False),
    ServingFramework('TorchServe',  'PyTorch models',                    'High',      True,  'Any',          False, None,  ['pytorch'],                 False),
    ServingFramework('Triton',      'Multi-framework, GPU optimization', 'Very High', True,  'Any',          False, None,  ['tensorflow','pytorch','onnx','tensorrt'], False),
    ServingFramework('Ray Serve',   'Python ML, composable pipelines',   'High',      True,  'Anyscale',     False, None,  ['any'],                     False),
    ServingFramework('MLflow',      'Experiment tracking + serving',     'Low-Med',   False, 'MLflow Cloud', False, None,  ['any'],                     False),
    ServingFramework('SageMaker',   'AWS managed, no K8s ops',           'Very High', True,  'AWS',          False, 'aws', ['any'],                     True),
    ServingFramework('Vertex AI',   'GCP managed, AutoML',               'Very High', True,  'GCP',          False, 'gcp', ['any'],                     True),
]

def recommend(cloud=None, framework=None, needs_gpu=False, k8s=False,
              needs_ab=False, needs_tracking=False):
    scored = []
    for fw in FRAMEWORKS:
        s, r = 0, []
        if cloud and fw.cloud == cloud:
            s += 50; r.append(f'native {cloud.upper()}')
        elif cloud and fw.cloud and fw.cloud != cloud:
            s -= 30
        if framework and (framework.lower() in fw.frameworks or 'any' in fw.frameworks):
            s += 10
            if framework.lower() in fw.frameworks and 'any' not in fw.frameworks:
                s += 15; r.append(f'optimized for {framework}')
        if needs_gpu and not fw.gpu: s -= 20
        if fw.requires_k8s and not k8s: s -= 25
        if needs_ab and fw.name in ['Seldon Core','KServe','SageMaker','Vertex AI']:
            s += 10; r.append('built-in A/B')
        if needs_tracking and fw.name == 'MLflow':
            s += 20; r.append('experiment tracking')
        if s > 0: scored.append((fw.name, s, '; '.join(r) or fw.best_for))
    return [(n, reason) for n, s, reason in sorted(scored, key=lambda x: -x[1])[:5]]

scenarios = [
    dict(cloud='aws', needs_gpu=True, k8s=False, label='AWS, no K8s, GPU'),
    dict(cloud='gcp', needs_gpu=True, k8s=True, needs_ab=True, label='GCP, K8s, GPU, A/B'),
    dict(framework='pytorch', needs_gpu=True, k8s=True, label='On-prem PyTorch, K8s'),
    dict(needs_tracking=True, label='Experiment tracking focus'),
]
for s in scenarios:
    label = s.pop('label')
    recs = recommend(**s)
    print(f'\nScenario: {label}')
    for i, (name, reason) in enumerate(recs, 1):
        print(f'  {i}. {name:15s} - {reason}')


Scenario: AWS, no K8s, GPU
  1. SageMaker       - native AWS

Scenario: GCP, K8s, GPU, A/B
  1. Vertex AI       - native GCP; built-in A/B
  2. KServe          - built-in A/B
  3. Seldon Core     - built-in A/B

Scenario: On-prem PyTorch, K8s
  1. KServe          - optimized for pytorch
  2. TorchServe      - optimized for pytorch
  3. Triton          - optimized for pytorch
  4. BentoML         - Python-first, quick deploy
  5. Seldon Core     - Inference graphs, A/B testing

Scenario: Experiment tracking focus
  1. MLflow          - experiment tracking


In [24]:
import math
from dataclasses import dataclass

@dataclass
class LatencyBudget:
    model_inference_ms: float
    batch_size: int
    network_ms: float
    preprocessing_ms: float
    postprocessing_ms: float = 0.0
    sla_ms: float = 200.0

def compute_budget(b):
    per_item_ms = b.model_inference_ms * math.sqrt(b.batch_size) / b.batch_size
    total = b.network_ms + b.preprocessing_ms + per_item_ms + b.postprocessing_ms
    throughputs = []
    for c in [1, 2, 4, 8, 16, 32]:
        svc  = per_item_ms + b.preprocessing_ms + b.postprocessing_ms
        wait = max(0, (c-1)*svc/2)
        throughputs.append((c, total + wait, c/((total + wait)/1000)))
    return {'network_ms': b.network_ms, 'preprocessing_ms': b.preprocessing_ms,
            'model_ms': per_item_ms, 'postprocessing_ms': b.postprocessing_ms,
            'total_ms': total, 'sla_ms': b.sla_ms, 'sla_ok': total <= b.sla_ms,
            'headroom_ms': b.sla_ms - total, 'throughputs': throughputs}

for title, b in [
    ('XGBoost, CPU, batch=1', LatencyBudget(5.0, 1, 15.0, 8.0, 1.0, 50.0)),
    ('BERT, GPU, batch=16',   LatencyBudget(80.0, 16, 5.0, 12.0, 2.0, 200.0)),
]:
    r = compute_budget(b)
    sep = '='*55
    print(f'\n{sep}\n{title}\n{sep}')
    print(f'  Network:        {r["network_ms"]:6.1f}ms')
    print(f'  Preprocessing:  {r["preprocessing_ms"]:6.1f}ms')
    print(f'  Model/item:     {r["model_ms"]:6.1f}ms')
    print(f'  Postprocessing: {r["postprocessing_ms"]:6.1f}ms')
    ok_str = 'OK' if r['sla_ok'] else 'BREACHED'
    print(f'  Total:          {r["total_ms"]:6.1f}ms (SLA {r["sla_ms"]}ms {ok_str})')
    print(f'  SLA headroom:   {r["headroom_ms"]:+.1f}ms')
    print(f'  {"Concurrency":>12s} {"Eff.Lat(ms)":>14s} {"QPS":>8s}')
    for c, lat, qps in r['throughputs']:
        print(f'  {c:>12d} {lat:>14.1f} {qps:>8.0f}')


XGBoost, CPU, batch=1


  Network:          15.0ms
  Preprocessing:     8.0ms
  Model/item:        5.0ms
  Postprocessing:    1.0ms
  Total:            29.0ms (SLA 50.0ms OK)
  SLA headroom:   +21.0ms
   Concurrency    Eff.Lat(ms)      QPS
             1           29.0       34
             2           36.0       56
             4           50.0       80
             8           78.0      103
            16          134.0      119
            32          246.0      130

BERT, GPU, batch=16
  Network:           5.0ms
  Preprocessing:    12.0ms
  Model/item:       20.0ms
  Postprocessing:    2.0ms
  Total:            39.0ms (SLA 200.0ms OK)
  SLA headroom:   +161.0ms
   Concurrency    Eff.Lat(ms)      QPS
             1           39.0       26
             2           56.0       36
             4           90.0       44
             8          158.0       51
            16          294.0       54
            32          566.0       57


## Additional Learning Resources

### Official Documentation
- [BentoML Documentation](https://docs.bentoml.com)
- [KServe Documentation](https://kserve.github.io/website/)
- [Seldon Core Documentation](https://docs.seldon.io/projects/seldon-core/en/latest/)
- [TensorFlow Serving](https://www.tensorflow.org/tfx/guide/serving)
- [TorchServe](https://pytorch.org/serve/)
- [Triton Inference Server](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/)
- [Ray Serve](https://docs.ray.io/en/latest/serve/index.html)
- [MLflow Models](https://mlflow.org/docs/latest/models.html)
- [Amazon SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html)
- [Vertex AI Prediction](https://cloud.google.com/vertex-ai/docs/predictions/overview)

### Key Papers
- Clipper: A Low-Latency Online Prediction Serving System (NSDI 2017)
- Serving DNNs like Clockwork (OSDI 2020)
- NVIDIA Triton Inference Server Best Practices